# Fine-Tuning Gemma for Financial Sentiment

In this notebook, we fine-tune **Google Gemma 3 1B** to classify financial news sentiment using **QLoRA** (quantized low-rank adaptation).

**Requirements:**
- Google Colab with a **T4 GPU** (free tier works). Go to Runtime → Change runtime type → T4 GPU.
- A **Hugging Face** account with an access token.
- Accept Google's **Gemma license** at [huggingface.co/google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) (click "Agree and access repository").

## Getting a Hugging Face Token

1. Create a free account at [huggingface.co](https://huggingface.co/join).
2. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. Click **Create new token**. Give it a name (e.g., "colab") and select **Read** access. Copy the token.

You have two options for using the token in this notebook:

**Option A — Colab Secrets (recommended):**
1. Click the 🔑 key icon in the left sidebar of Colab.
2. Click **Add new secret**. Name it `HF_TOKEN` and paste your token as the value.
3. Toggle "Notebook access" on.

**Option B — Paste directly:**
Replace `None` in the cell below with your token as a string, e.g., `"hf_abc123..."`.

## Install Libraries

In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

## Authenticate with Hugging Face

This cell tries to read `HF_TOKEN` from Colab Secrets first. If that fails, it falls back to the `hf_token` variable below. Set one or the other.

In [ ]:
# Option B: paste your token here if not using Colab Secrets
hf_token = None  # e.g., "hf_abc123..."

# Try Colab Secrets first
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token is None:
    raise ValueError(
        "No HF token found. Either add HF_TOKEN to Colab Secrets "
        "or set hf_token = 'hf_...' in the cell above."
    )

from huggingface_hub import login
login(token=hf_token)

## Load the Model with QLoRA

We load Gemma 3 1B with 4-bit quantization. This compresses the model from ~2 GB to ~800 MB, making it possible to fine-tune on a free Colab T4 (16 GB VRAM).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)

## Attach LoRA Adapters

Instead of updating all ~1 billion parameters, LoRA freezes the original weights and trains small adapter matrices on the attention layers. This reduces trainable parameters to < 0.1% of the total.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                          # rank — higher = more capacity
    lora_alpha=32,                 # scaling factor
    target_modules=["q_proj", "v_proj"],  # attention layers only
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Prepare Training Data

We use **Financial PhraseBank** — 4,845 sentences from financial news, each labeled as positive, negative, or neutral by finance experts. We format each example as a chat conversation so the model learns to respond with just the sentiment label.

In [ ]:
from datasets import load_dataset

# Load Financial PhraseBank — 4,845 sentences labeled by finance experts
dataset = load_dataset("financial_phrasebank", "sentences_allagree",
                       split="train", trust_remote_code=True)

def format_example(example):
    labels = {0: "negative", 1: "neutral", 2: "positive"}
    text = example["sentence"]
    label = labels[example["label"]]
    messages = [
        {"role": "user", "content": f"Classify the financial sentiment: {text}"},
        {"role": "assistant", "content": label},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_example)

# Hold out 10% for testing
split = dataset.train_test_split(test_size=0.1, seed=42)
train_data, test_data = split["train"], split["test"]

## Train

We use the `SFTTrainer` (supervised fine-tuning) from the `trl` library. On a Colab T4 GPU, training takes roughly 10–15 minutes.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./gemma-finance-sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    max_seq_length=256,
    logging_steps=10,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
)

trainer.train()

## Test the Result

Let's test on a sentence the model has never seen.

In [ ]:
messages = [
    {"role": "user",
     "content": "Classify the financial sentiment: "
                "Net income fell 12% as rising input costs squeezed margins."},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt",
                                       add_generation_prompt=True)
inputs = inputs.to(model.device)
outputs = model.generate(inputs, max_new_tokens=5)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:],
                       skip_special_tokens=True))

## Save the Adapter

The LoRA adapter is only ~14 MB. You can save it and later reload it on top of the base model, or merge it into the base weights for a single model file.

In [ ]:
# Save the LoRA adapter (just a few MB)
model.save_pretrained("./gemma-finance-sentiment")

# Later, reload on any machine:
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(model_id, ...)
# model = PeftModel.from_pretrained(base, "./gemma-finance-sentiment")

# Or merge into a single model file:
# merged = model.merge_and_unload()
# merged.save_pretrained("./gemma-finance-merged")